# DSSATLab walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdelrahmanAmr3/dssatlab/blob/master/notebook/dssatlab_walkthrough.ipynb)

A hands-on tour of `dssatlab`, using the latest code from GitHub (before the next PyPI release):

1. `connect()` finds DSSAT and `install()` builds it on Linux/Colab (sections 1-4).
2. `Simulation` runs one treatment of a FileX with **your own weather data** (sections 5-7).
3. The same `Simulation` with your own **soil** and **management** data (sections 8-9).

Run the cells top to bottom. The build in section 2 takes several minutes. On a machine that already has DSSAT (for example Windows), skip section 2 and run the rest.

## Setup

In [ ]:
%pip install -q "git+https://github.com/AbdelrahmanAmr3/dssatlab.git"

In [ ]:
import logging
import dssatlab as dl

# dssatlab logs how it finds DSSAT; show those messages here.
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
print("dssatlab", dl.__version__)

## 1. `connect()` before installing

On a fresh runtime DSSAT is not installed, so we expect `DSSATNotFoundError`. It lists what was checked and what to do next.
`interactive=False` raises the error instead of asking a question (normally `dl.connect()` would offer to install).

In [ ]:
try:
    print(dl.connect(interactive=False))
except dl.DSSATNotFoundError as e:
    print(e)

## 2. `install()`

Downloads the latest stable DSSAT release from GitHub and builds it with git, cmake and gfortran (already on Colab).

In [ ]:
import time

t0 = time.time()
dssat = dl.install()
print(dssat)
print(f"took {time.time() - t0:.0f}s")

## 3. `connect()` after installing

In [ ]:
# The build was remembered, so connect() now returns it straight away.
assert dl.connect(interactive=False) == dssat

## 4. `install()` again

The build is reused, so this should take a moment instead of minutes.

In [ ]:
t0 = time.time()
again = dl.install()
print(again, f"({time.time() - t0:.1f}s)")
assert again == dssat

## 5. Get a sample FileX

`Simulation` needs a FileX and the genotype files beside it. We download DSSAT's own Gainesville maize experiment (`UFGA8201.MZX`) from the official DSSAT repositories into a folder called `demo`. Its weather file is downloaded too, but only so we can turn it into a CSV in the next section; the `Simulation` does not use it.

In [ ]:
import urllib.request
from pathlib import Path

DATA = "https://raw.githubusercontent.com/DSSAT/dssat-csm-os/develop/Data"
EXPERIMENTS = "https://raw.githubusercontent.com/DSSAT/dssat-csm-data/develop"
demo = Path("demo")
demo.mkdir(exist_ok=True)

files = {
    "UFGA8201.MZX": f"{EXPERIMENTS}/Maize/UFGA8201.MZX",
    "UFGA8201.WTH": f"{EXPERIMENTS}/Weather/UFGA8201.WTH",
    "MZCER048.CUL": f"{DATA}/Genotype/MZCER048.CUL",
    "MZCER048.ECO": f"{DATA}/Genotype/MZCER048.ECO",
    "MZCER048.SPE": f"{DATA}/Genotype/MZCER048.SPE",
}
for name, url in files.items():
    if not (demo / name).exists():
        urllib.request.urlretrieve(url, demo / name)
print(sorted(p.name for p in demo.iterdir()))

## 6. Your weather data as a CSV

`Simulation` reads daily weather from a CSV with fixed column names. `write_weather_template()` writes an example you can edit. Here we skip the editing: we convert the sample `.WTH` into that CSV layout, so you can see what real data looks like.

In [ ]:
import csv

dl.write_weather_template("weather_template.csv")
print(Path("weather_template.csv").read_text().splitlines()[:3], "...")

In [ ]:
# Turn the sample UFGA8201.WTH into a weather CSV (this is the part you would do with your own data).
lines = (demo / "UFGA8201.WTH").read_text().splitlines()
site = lines[lines.index(next(l for l in lines if l.startswith("@ INSI"))) + 1].split()
start = next(i for i, l in enumerate(lines) if l.startswith("@DATE")) + 1

from datetime import date, timedelta

rows = []
for line in lines[start:]:
    if not line.strip():
        continue
    d, srad, tmax, tmin, rain = line.split()[:5]
    day = date(1900 + int(d[:2]), 1, 1) + timedelta(days=int(d[2:]) - 1)
    rows.append({"station": site[0], "latitude": site[1], "longitude": site[2],
                 "elevation": site[3], "date": day.isoformat(), "srad": srad,
                 "tmax": tmax, "tmin": tmin, "rain": rain})

with open("weather.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(len(rows), "days:", rows[0]["date"], "to", rows[-1]["date"])

## 7. Check and run a `Simulation`

`Simulation(...)` only stores its inputs. `check()` reads the weather and FileX and returns a list of problems without writing anything; an empty list means it is ready. `run()` repeats the checks, copies the FileX and genotype files into a fresh `dssat_sim_...` folder, writes the weather file, and runs DSSAT there. Your originals are left untouched.

In [ ]:
sim = dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather="weather.csv")
problems = sim.check(verbose=True)
print("problems:", problems)

In [ ]:
result = sim.run()
print("exit status:", result.returncode)
print("run directory:", result.run_dir)
print([p.name for p in result.outputs])

### Look at the output

`Summary.OUT` has one line per treatment run. `result.summary()` reads it into a list of dicts with DSSAT's own column names (`HWAM`, `MDAT`, ...).

In [ ]:
row = result.summary()[0]
print("Harvested grain yield (HWAM, kg/ha):", row["HWAM"])
print("Maturity date (MDAT, YYYYDDD):     ", row["MDAT"])

### What a mistake looks like

Here the weather station code does not match the FileX's `WSTA` (`UFGA`). `check()` reports it, and `run()` refuses to start DSSAT.

In [ ]:
bad = [dict(row, station="XXXX") for row in rows]
try:
    dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather=bad).run()
except dl.DSSATCheckError as error:
    for problem in error.problems:
        print("-", problem)

## 8. Your own soil data

Pass `soil=` and `dssatlab` writes a `SOIL.SOL` from it instead of using a soil file. The soil is one profile, one row per layer, and its `soil_id` must equal the FileX's `ID_SOIL` (`IBMZ910014` for treatment 1). We start from the template and change the ID.

In [ ]:
dl.write_soil_template("soil_template.csv")
with open("soil_template.csv", newline="") as f:
    soil = [dict(row, soil_id="IBMZ910014") for row in csv.DictReader(f)]
print(len(soil), "layers, soil_id", soil[0]["soil_id"])

sim = dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather="weather.csv", soil=soil)
print("problems:", sim.check())
result_soil = sim.run()
print(result_soil.run_dir)

## 9. Your own management data

Pass `management=` to replace planting, irrigation or fertilizer for the selected treatment. It can be a YAML file (`write_management_template()` writes a documented one, needs `pip install pyyaml`) or a plain dict as below. Dates are quoted `"YYYY-MM-DD"` strings. Here we add one irrigation on top of treatment 1.

In [ ]:
management = {"treatments": {1: {"irrigation": [
    {"date": "1982-03-15", "amount": 30.0, "method": "IR001"},
]}}}

sim = dl.Simulation(demo / "UFGA8201.MZX", treatment=1, weather="weather.csv",
                    management=management)
print("problems:", sim.check())   # management given, so the report is printed
result_mgmt = sim.run()
print(result_mgmt.run_dir)

## 10. Compare

Each run wrote its own `Summary.OUT`. The yields (`HWAM`) differ because the soil and irrigation differ. Note that the soil is the template's example profile, not a real soil for Gainesville, so this shows that the inputs are used, not which yield is right.

In [ ]:
for label, res in [("FileX defaults", result), ("own soil", result_soil), ("irrigation added", result_mgmt)]:
    print(f"{label:18s} HWAM = {res.summary()[0]['HWAM']} kg/ha")

## 11. Reading results

Run the sample maize FileX for all six treatments: rainfed, irrigated and vegetative stress, each with low or high nitrogen. We copy the sample folder so the installed files stay untouched.

In a verified sample run, rainfed yield was 2293 kg/ha at both nitrogen levels; irrigated yield was 8207 kg/ha with low nitrogen and 11854 kg/ha with high nitrogen.

For plots, install `pip install dssatlab[plot]`. Both plots below compare treatments from the same run directory. Pandas is optional: install `pip install pandas` to run the final DataFrame cell, or skip it.


In [ ]:
from pathlib import Path
import shutil
import tempfile

sample_dir = dssat.parent / "Maize"
work_dir = Path(tempfile.mkdtemp(prefix="dssatlab_results_"))
maize_dir = work_dir / "Maize"
shutil.copytree(sample_dir, maize_dir)

# Omitting treatment runs all six treatments in this FileX.
result = dl.run(maize_dir / "UFGA8201.MZX", executable=dssat)
print(result.run_dir)


In [ ]:
for row in result.summary():
    print(f"{row['TNAM']}: {row['HWAM']} kg/ha")


In [ ]:
dl.plot_plant_growth(result.run_dir, "LAID")


In [ ]:
result.plot("CWAD")


In [ ]:
dl.to_dataframe(result.summary())
